In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score

import os
from pathlib import Path

# 1. Carga de archivos: carpeta de la ciudad en qa/ (CIUDAD=merida → qa/Bepensa · CIUDAD=guadalajara → qa/Guadalajara)
CIUDAD = os.environ.get("CIUDAD", "merida")
QA = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()) / "qa" / {"merida": "Bepensa", "guadalajara": "Guadalajara"}[CIUDAD]
file_model = next(QA.glob("06_letras_nse_ventas_tradicional_*.xlsx"))
print(f"{CIUDAD}: {file_model.relative_to(QA.parent.parent)}")

# Cargar Modelo Nuevo (Canal Tradicional de la ZM)
xls_model = pd.ExcelFile(file_model)
raw_model = pd.read_excel(
    xls_model, sheet_name='Letras por PDV', header=None
)
df_model = raw_model.iloc[5:].copy()
df_model.columns = raw_model.iloc[4].values

# Limpieza y filtrado de PDVs objetivo
df_model['pos_id'] = pd.to_numeric(df_model['pos_id'], errors='coerce')
df_model = df_model[df_model['pos_id'].notna()].copy()
df_model['venta_media_num'] = pd.to_numeric(
    df_model['Venta media (cajas/mes)'], errors='coerce'
)

df_obj = df_model[
    (df_model['venta_media_num'] > 0)
    & (
        df_model['Letra 1 sin AltScore']
        != 'sin NSE (sin hogares a 300 m)'
    )
].copy()

# 2. Cálculo de Accuracy Global y Kappa
y_base = df_obj['Letras base (INEGI + CP)']
y_final = df_obj['Letras (final)']

acc_total = accuracy_score(y_base, y_final)
kappa_total = cohen_kappa_score(y_base, y_final)

print("=== ACCURACY MODELO NUEVO VS MODELO ANTERIOR (BASE) ===")
print(f"Accuracy Total: {acc_total * 100:.2f}%")
print(f"Cohen's Kappa (κ): {kappa_total:.3f}\n")

# Matriz de Confusión / Transición
cm_total = pd.crosstab(
    y_base,
    y_final,
    rownames=['Anterior (Base)'],
    colnames=['Nuevo (Final)'],
    margins=True,
)
print("Matriz de Transición:")
print(cm_total)

# 3. Accuracy individual Letra 1 y Letra 2
acc_l1 = accuracy_score(
    df_obj['Letra 1 sin AltScore'], df_obj['Letra 1 con AltScore (final)']
)
kappa_l1 = cohen_kappa_score(
    df_obj['Letra 1 sin AltScore'], df_obj['Letra 1 con AltScore (final)']
)

acc_l2 = accuracy_score(
    df_obj['Letra 2 sin Rappi'], df_obj['Letra 2 con Rappi (final)']
)
kappa_l2 = cohen_kappa_score(
    df_obj['Letra 2 sin Rappi'], df_obj['Letra 2 con Rappi (final)']
)

print("\n=== DESGLOSE POR COMPONENTE DE LETRA ===")
print(f"Accuracy Letra 1 (NSE / Entorno): {acc_l1 * 100:.2f}% (κ = {kappa_l1:.3f})")
print(f"Accuracy Letra 2 (Ventas / Desempeño): {acc_l2 * 100:.2f}% (κ = {kappa_l2:.3f})")

guadalajara: qa\Guadalajara\06_letras_nse_ventas_tradicional_arca_zm_guadalajara.xlsx


=== ACCURACY MODELO NUEVO VS MODELO ANTERIOR (BASE) ===
Accuracy Total: 84.06%
Cohen's Kappa (κ): 0.777

Matriz de Transición:
Nuevo (Final)      HH    HL    LH    LL    All
Anterior (Base)                               
HH               2967   316   416    61   3760
HL                283  6786    17   873   7959
LH                404    55  3205   455   4119
LL                 15   945    72  7678   8710
All              3669  8102  3710  9067  24548

=== DESGLOSE POR COMPONENTE DE LETRA ===
Accuracy Letra 1 (NSE / Entorno): 88.65% (κ = 0.773)
Accuracy Letra 2 (Ventas / Desempeño): 94.81% (κ = 0.879)
